# Subjective thermo-physiological questionnaire analysis

This notebook processes the repeated subjective responses collected during the three ventilation conditions. It preserves ordinal responses and produces descriptive summaries and figures. No additional inferential analysis is run in the current configuration.

Expected input: `data/raw/subjective_questionnaires_restructured.xlsx`.


## 1. Imports and analysis configuration

The confirmed coding is: condition 0 = no ventilation, 1 = low ventilation and 2 = high ventilation; questionnaire timepoints run from 0 to 50 min at 5-min intervals. `RUN_ORDINAL_INFERENCE` is kept set to `False`.


In [1]:
"""Analyse repeated subjective thermo-physiological questionnaire responses.

Expected input
--------------
data/raw/subjective_questionnaires_restructured.xlsx

The workbook is produced from the original Google Forms export and contains:
    - subjective_long
    - recovery_long
    - post_trial

Important
---------
The condition-code mapping and the timing of the 11 main questionnaire blocks
must be confirmed against the experimental log before phase-level inference is
treated as final. The raw response columns are never overwritten.
"""

from __future__ import annotations

from itertools import product
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd


# ============================================================
# CONFIGURATION TO CONFIRM AGAINST THE EXPERIMENTAL LOG
# ============================================================

CONDITION_CODE_MAP = {
    0: "no_ventilation",
    1: "low_ventilation",
    2: "high_ventilation",
}

CONDITION_ORDER = [
    "no_ventilation",
    "low_ventilation",
    "high_ventilation",
]

CONDITION_LABELS = {
    "no_ventilation": "No ventilation",
    "low_ventilation": "Low ventilation",
    "high_ventilation": "High ventilation",
}

CONDITION_COLORS = {
    "no_ventilation": "blue",
    "low_ventilation": "red",
    "high_ventilation": "black",
}

# Provisional reconstruction suggested by the questionnaire structure:
# 11 blocks at 0, 5, ..., 50 min. Do not set this flag to True until the
# experimental log confirms both the minutes and their phase assignment.
CONDITION_MAPPING_CONFIRMED = True
TIMEPOINT_MAPPING_CONFIRMED = True

# The subjective results are retained as a complementary descriptive analysis.
# No additional ordinal inferential models are run unless explicitly requested.
RUN_ORDINAL_INFERENCE = False

TIMEPOINT_METADATA = pd.DataFrame(
    {
        "timepoint_index": list(range(1, 12)),
        "elapsed_min": [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50],
        "phase": [
            "standing_1",
            "standing_1",
            "standing_1",
            "walk_low_1",
            "walk_low_1",
            "walk_mod_1",
            "walk_mod_1",
            "walk_low_2",
            "walk_low_2",
            "walk_mod_2",
            "walk_mod_2",
        ],
    }
)

PHASE_ORDER = [
    "standing_1",
    "walk_low_1",
    "walk_mod_1",
    "walk_low_2",
    "walk_mod_2",
    "standing_2",
]

PHASE_LABELS = {
    "standing_1": "Standing 1",
    "walk_low_1": "Walk low 1",
    "walk_mod_1": "Walk moderate 1",
    "walk_low_2": "Walk low 2",
    "walk_mod_2": "Walk moderate 2",
    "standing_2": "Standing 2",
}

PRIMARY_OUTCOMES = {
    "thermal_global_score": "Global thermal sensation",
    "thermal_trunk_score": "Trunk thermal sensation",
    "wetness_global_score": "Global wetness sensation",
    "wetness_trunk_score": "Trunk wetness sensation",
    "acceptability_score": "Thermal acceptability",
}



## 2. Input paths and analysis functions


In [2]:
def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root without relying on a machine-specific path."""
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (
            (candidate / ".git").exists()
            or (candidate / "reports").exists()
        ):
            return candidate
    raise RuntimeError(
        "Project root not found. Run this script inside the "
        "convective_cooling repository."
    )


# Prefer the path used in the WSL project. The fallback keeps the analysis
# portable when the repository is moved or executed on another machine.
PREFERRED_PROJECT_ROOT = Path(
    "/home/eleonorab/repository/convective_cooling"
)
ROOT = (
    PREFERRED_PROJECT_ROOT
    if PREFERRED_PROJECT_ROOT.exists()
    else find_project_root()
)
INPUT_FILE = (
    ROOT
    / "data"
    / "raw"
    / "subjective_questionnaires_restructured.xlsx"
)
OUTPUT_DIR = (
    ROOT
    / "reports"
    / "07_subjective_questionnaires"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Questionnaire workbook expected at:", INPUT_FILE)


def require_columns(table: pd.DataFrame, columns: list[str], name: str) -> None:
    missing = [column for column in columns if column not in table.columns]
    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")


def validate_condition_mapping(*tables: pd.DataFrame) -> None:
    """Ensure the workbook agrees with the configured condition mapping."""
    for index, table in enumerate(tables, start=1):
        expected = table["condition_code"].map(CONDITION_CODE_MAP)
        mismatch = expected.ne(table["condition"])
        if mismatch.any():
            examples = table.loc[
                mismatch,
                ["trial_id", "condition_code", "condition"],
            ].head()
            raise ValueError(
                f"Condition mapping mismatch in table {index}:\n"
                f"{examples.to_string(index=False)}"
            )


def validate_balanced_trials(table: pd.DataFrame) -> None:
    trials = table[["subject_id", "condition"]].drop_duplicates()
    counts = trials.groupby("subject_id")["condition"].nunique()
    if len(counts) != 10 or not counts.eq(3).all():
        raise ValueError(
            "Expected 10 participants with all three conditions. "
            f"Observed counts:\n{counts.to_string()}"
        )


def add_confirmed_time_metadata(table: pd.DataFrame) -> pd.DataFrame:
    """Attach time and phase only after explicit confirmation."""
    table = table.drop(columns=["elapsed_min", "phase"], errors="ignore")
    if not TIMEPOINT_MAPPING_CONFIRMED:
        table["elapsed_min"] = np.nan
        table["phase"] = pd.NA
        return table
    return table.merge(
        TIMEPOINT_METADATA,
        on="timepoint_index",
        how="left",
        validate="many_to_one",
    )


def ordinal_summary(
    table: pd.DataFrame,
    group_columns: list[str],
    outcome: str,
) -> pd.DataFrame:
    """Median and IQR; suitable descriptive summaries for ordinal scores."""
    return (
        table.dropna(subset=[outcome])
        .groupby(group_columns, observed=True)[outcome]
        .agg(
            n="count",
            median="median",
            q25=lambda values: values.quantile(0.25),
            q75=lambda values: values.quantile(0.75),
        )
        .reset_index()
    )


def category_proportions(
    table: pd.DataFrame,
    group_columns: list[str],
    outcome: str,
) -> pd.DataFrame:
    """Return counts and proportions for every observed ordinal category."""
    clean = table.dropna(subset=[outcome]).copy()
    counts = (
        clean.groupby(group_columns + [outcome], observed=True)
        .size()
        .rename("n")
        .reset_index()
    )
    totals = (
        counts.groupby(group_columns, observed=True)["n"]
        .transform("sum")
    )
    counts["proportion"] = counts["n"] / totals
    return counts


def plot_ordinal_trajectory(
    summary: pd.DataFrame,
    x_column: str,
    outcome_label: str,
    filename: str,
    tick_labels: dict | None = None,
) -> None:
    fig, ax = plt.subplots(figsize=(10, 5.2), dpi=300)
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    x_values = list(dict.fromkeys(summary[x_column].dropna().tolist()))
    x_position = {value: index for index, value in enumerate(x_values)}

    for condition in CONDITION_ORDER:
        subset = summary.loc[summary["condition"] == condition].copy()
        subset["x"] = subset[x_column].map(x_position)
        subset = subset.sort_values("x")
        if subset.empty:
            continue

        median = subset["median"].to_numpy(dtype=float)
        lower = median - subset["q25"].to_numpy(dtype=float)
        upper = subset["q75"].to_numpy(dtype=float) - median

        q25 = median - lower
        q75 = median + upper
        color = CONDITION_COLORS[condition]

        ax.fill_between(
            subset["x"].to_numpy(dtype=float),
            q25,
            q75,
            color=color,
            alpha=0.14,
            linewidth=0,
            zorder=1,
        )
        ax.plot(
            subset["x"],
            median,
            marker="o",
            markersize=6,
            markerfacecolor=color,
            markeredgecolor="white",
            markeredgewidth=0.8,
            linewidth=2.0,
            color=color,
            label=CONDITION_LABELS[condition],
            zorder=3,
        )

    labels = [tick_labels.get(value, value) if tick_labels else value for value in x_values]
    ax.set_xticks(range(len(x_values)))
    ax.set_xticklabels(labels, rotation=20, ha="right")
    if x_column == "elapsed_min":
        ax.set_xlabel("Elapsed time [min]", fontsize=16)
    elif x_column == "timepoint_index":
        ax.set_xlabel("Questionnaire timepoint", fontsize=16)
    else:
        ax.set_xlabel("Experimental phase", fontsize=16)
    ax.set_ylabel(f"{outcome_label}\nmedian [IQR]", fontsize=16)
    ax.legend(
        frameon=False,
        ncol=3,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.12),
        columnspacing=1.8,
        handlelength=2.4,
    )
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(axis="y", color="#D9D9D9", linewidth=0.8, alpha=0.55)
    ax.set_axisbelow(True)

    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_visible(True)
    ax.spines["bottom"].set_color("black")
    ax.spines["bottom"].set_linewidth(2.5)
    ax.tick_params(axis="y", length=0, labelsize=14)
    ax.tick_params(
        axis="x",
        length=6,
        width=2.0,
        color="black",
        labelsize=14,
    )
    ax.margins(x=0.02)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, bbox_inches="tight")
    plt.close(fig)


def fit_ordinal_gee(
    table: pd.DataFrame,
    outcome: str,
    model_name: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Fit a population-averaged ordinal model with participant clustering.

    This uses an ordinal GEE rather than treating ordinal scores as continuous.
    With only 10 participants, results must be interpreted cautiously and the
    descriptive distributions should remain visible alongside p-values.
    """
    try:
        from statsmodels.genmod.cov_struct import Exchangeable
        from statsmodels.genmod.generalized_estimating_equations import OrdinalGEE
    except ImportError as error:
        raise ImportError(
            "Ordinal inference requires statsmodels with OrdinalGEE. "
            "Install or update statsmodels in the analysis environment."
        ) from error

    analysis = table.dropna(
        subset=[outcome, "condition", "phase", "subject_id"]
    ).copy()
    analysis[outcome] = analysis[outcome].astype(int)

    formula = (
        f"{outcome} ~ "
        "C(condition, Treatment(reference='no_ventilation')) * "
        "C(phase, Treatment(reference='standing_1'))"
    )
    model = OrdinalGEE.from_formula(
        formula,
        groups="subject_id",
        data=analysis,
        cov_struct=Exchangeable(),
    )
    result = model.fit(maxiter=200)

    coefficient_table = pd.DataFrame(
        {
            "term": result.params.index,
            "estimate": result.params.to_numpy(),
            "standard_error": result.bse.to_numpy(),
            "z": result.tvalues.to_numpy(),
            "p_value": result.pvalues.to_numpy(),
            "ci_low": result.conf_int().iloc[:, 0].to_numpy(),
            "ci_high": result.conf_int().iloc[:, 1].to_numpy(),
        }
    )
    coefficient_table.insert(0, "model", model_name)

    interaction_terms = [
        name for name in result.params.index
        if ":" in name
    ]
    if interaction_terms:
        restriction = np.zeros(
            (len(interaction_terms), len(result.params)),
            dtype=float,
        )
        parameter_names = list(result.params.index)
        for row_index, term in enumerate(interaction_terms):
            restriction[row_index, parameter_names.index(term)] = 1.0
        test = result.wald_test(restriction, scalar=True)
        global_test = pd.DataFrame(
            {
                "model": [model_name],
                "test": ["condition_by_phase_interaction"],
                "statistic": [float(np.asarray(test.statistic).squeeze())],
                "df": [len(interaction_terms)],
                "p_value": [float(np.asarray(test.pvalue).squeeze())],
                "n_observations": [len(analysis)],
                "n_participants": [analysis["subject_id"].nunique()],
            }
        )
    else:
        global_test = pd.DataFrame()

    return coefficient_table, global_test



Project root: /home/eleonorab/repository/convective_cooling
Questionnaire workbook expected at: /home/eleonorab/repository/convective_cooling/data/raw/subjective_questionnaires_restructured.xlsx


## 3. Analysis workflow

The workflow validates the repeated-measures design, creates ordinal summaries and category proportions, and describes the 52–60 min recovery responses.


In [3]:
def main() -> None:
    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"Input workbook not found: {INPUT_FILE}\n"
            "Copy subjective_questionnaires_restructured.xlsx to data/raw."
        )

    subjective = pd.read_excel(INPUT_FILE, sheet_name="subjective_long")
    recovery = pd.read_excel(INPUT_FILE, sheet_name="recovery_long")
    post_trial = pd.read_excel(INPUT_FILE, sheet_name="post_trial")

    require_columns(
        subjective,
        [
            "trial_id", "subject_id", "condition_code", "condition",
            "timepoint_index", *PRIMARY_OUTCOMES.keys(),
            "air_movement_score", "rpe_score",
        ],
        "subjective_long",
    )
    require_columns(
        recovery,
        [
            "trial_id", "subject_id", "condition_code", "condition",
            "elapsed_min", "thermal_score", "wetness_score",
        ],
        "recovery_long",
    )
    require_columns(
        post_trial,
        [
            "trial_id", "subject_id", "condition_code", "condition",
            "fan_noise_raw", "garment_weight_perception_raw",
            "garment_rating",
        ],
        "post_trial",
    )

    validate_condition_mapping(subjective, recovery, post_trial)
    validate_balanced_trials(subjective)
    subjective = add_confirmed_time_metadata(subjective)

    quality = pd.DataFrame(
        {
            "check": [
                "participants",
                "trials",
                "main_questionnaire_rows",
                "recovery_rows",
                "condition_mapping_confirmed_in_script",
                "timepoint_mapping_confirmed_in_script",
            ],
            "value": [
                subjective["subject_id"].nunique(),
                subjective["trial_id"].nunique(),
                len(subjective),
                len(recovery),
                CONDITION_MAPPING_CONFIRMED,
                TIMEPOINT_MAPPING_CONFIRMED,
            ],
        }
    )
    quality.to_csv(OUTPUT_DIR / "questionnaire_quality_checks.csv", index=False)

    # Descriptions are displayed against elapsed time once the mapping is confirmed.
    descriptive_x = (
        "elapsed_min"
        if TIMEPOINT_MAPPING_CONFIRMED
        else "timepoint_index"
    )
    timepoint_outputs = []
    category_outputs = []
    for outcome, label in PRIMARY_OUTCOMES.items():
        summary = ordinal_summary(
            subjective,
            ["condition", descriptive_x],
            outcome,
        )
        summary.insert(0, "outcome", outcome)
        timepoint_outputs.append(summary)

        proportions = category_proportions(
            subjective,
            ["condition", descriptive_x],
            outcome,
        )
        proportions.insert(0, "outcome", outcome)
        category_outputs.append(proportions)

        plot_ordinal_trajectory(
            summary,
            x_column=descriptive_x,
            outcome_label=label,
            filename=f"{outcome}_by_timepoint.png",
        )

    pd.concat(timepoint_outputs, ignore_index=True).to_csv(
        OUTPUT_DIR / "subjective_timepoint_descriptives.csv",
        index=False,
    )
    pd.concat(category_outputs, ignore_index=True).to_csv(
        OUTPUT_DIR / "subjective_timepoint_category_proportions.csv",
        index=False,
    )

    # Manipulation checks: perceived airflow and perceived exertion.
    manipulation = []
    for outcome in ["air_movement_score", "rpe_score"]:
        current = ordinal_summary(
            subjective,
            ["condition", descriptive_x],
            outcome,
        )
        current.insert(0, "outcome", outcome)
        manipulation.append(current)
    pd.concat(manipulation, ignore_index=True).to_csv(
        OUTPUT_DIR / "manipulation_check_descriptives.csv",
        index=False,
    )

    # Standing 2 is known exactly from the recovery questions.
    recovery_outputs = []
    recovery_categories = []
    for outcome, label in {
        "thermal_score": "Recovery thermal sensation",
        "wetness_score": "Recovery wetness sensation",
    }.items():
        summary = ordinal_summary(
            recovery,
            ["condition", "elapsed_min"],
            outcome,
        )
        summary.insert(0, "outcome", outcome)
        recovery_outputs.append(summary)

        proportions = category_proportions(
            recovery,
            ["condition", "elapsed_min"],
            outcome,
        )
        proportions.insert(0, "outcome", outcome)
        recovery_categories.append(proportions)

        plot_ordinal_trajectory(
            summary,
            x_column="elapsed_min",
            outcome_label=label,
            filename=f"{outcome}_standing_2.png",
        )

    pd.concat(recovery_outputs, ignore_index=True).to_csv(
        OUTPUT_DIR / "standing2_recovery_descriptives.csv",
        index=False,
    )
    pd.concat(recovery_categories, ignore_index=True).to_csv(
        OUTPUT_DIR / "standing2_recovery_category_proportions.csv",
        index=False,
    )

    # End-of-trial acceptability is kept descriptive.
    post_summary = []
    for variable in [
        "fan_noise_raw",
        "garment_weight_perception_raw",
        "garment_rating",
    ]:
        counts = (
            post_trial.groupby(["condition", variable], dropna=False)
            .size()
            .rename("n")
            .reset_index()
        )
        counts.insert(0, "outcome", variable)
        post_summary.append(counts)
    pd.concat(post_summary, ignore_index=True).to_csv(
        OUTPUT_DIR / "post_trial_descriptives.csv",
        index=False,
    )

    if RUN_ORDINAL_INFERENCE and TIMEPOINT_MAPPING_CONFIRMED:
        coefficient_tables = []
        global_tests = []
        for outcome in [
            "thermal_trunk_score",
            "wetness_trunk_score",
            "acceptability_score",
        ]:
            coefficients, test = fit_ordinal_gee(
                subjective,
                outcome,
                model_name=outcome,
            )
            coefficient_tables.append(coefficients)
            global_tests.append(test)

        pd.concat(coefficient_tables, ignore_index=True).to_csv(
            OUTPUT_DIR / "ordinal_gee_coefficients.csv",
            index=False,
        )
        pd.concat(global_tests, ignore_index=True).to_csv(
            OUTPUT_DIR / "ordinal_gee_global_tests.csv",
            index=False,
        )
    elif RUN_ORDINAL_INFERENCE:
        warnings.warn(
            "TIMEPOINT_MAPPING_CONFIRMED is False. Descriptive outputs were "
            "created, but phase-level ordinal models were intentionally skipped."
        )

    print(f"Input: {INPUT_FILE}")
    print(f"Outputs: {OUTPUT_DIR}")
    print(f"Participants: {subjective['subject_id'].nunique()}")
    print(f"Trials: {subjective['trial_id'].nunique()}")
    print(f"Main responses: {len(subjective)}")
    print(f"Recovery responses: {len(recovery)}")
    print(
        "Phase inference run:",
        RUN_ORDINAL_INFERENCE and TIMEPOINT_MAPPING_CONFIRMED,
    )



## 4. Run the analysis

If the input file is missing, the error message below reports the exact path that must contain the workbook.


In [4]:
main()


Input: /home/eleonorab/repository/convective_cooling/data/raw/subjective_questionnaires_restructured.xlsx
Outputs: /home/eleonorab/repository/convective_cooling/reports/07_subjective_questionnaires
Participants: 10
Trials: 30
Main responses: 330
Recovery responses: 150
Phase inference run: False
